환경

코드

추론

In [1]:
config_dict = {
    "loss_scales": {
        "rec": 1.0, "rew": 1.0, "con": 1.0, "dyn": 1.0,
        "rep": 0.1, "policy": 1.0, "value": 1.0, "repval": 0.3
    },
    "opt": {
        "lr": 4e-05, "agc": 0.3, "eps": 1e-20, "beta1": 0.9,
        "beta2": 0.999, "momentum": True, "wd": 0.0,
        "schedule": "const", "warmup": 1000, "anneal": 0
    },
    "ac_grads": False,
    "dyn": {
        "typ": "rssm",
        "rssm": {
            "deter": 2048, "hidden": 256, "stoch": 32, "classes": 16,
            "act": "silu", "norm": "rms", "unimix": 0.01,
            "outscale": 1.0, "winit": "trunc_normal_in",
            "imglayers": 2, "obslayers": 1, "dynlayers": 1,
            "absolute": False, "blocks": 8, "free_nats": 1.0
        }
    },
    "enc": {
        "typ": "simple",
        "simple": {
            "depth": 16, "mults": [2, 3, 4, 4], "layers": 3,
            "units": 256, "act": "silu", "norm": "rms",
            "winit": "trunc_normal_in", "symlog": True,
            "outer": False, "kernel": 5, "strided": False
        }
    },
    "dec": {
        "typ": "simple",
        "simple": {
            "depth": 16, "mults": [2, 3, 4, 4], "layers": 3,
            "units": 256, "act": "silu", "norm": "rms",
            "outscale": 1.0, "winit": "trunc_normal_in",
            "outer": False, "kernel": 5, "bspace": 8, "strided": False
        }
    },
    "rewhead": {
        "layers": 1, "units": 256, "act": "silu", "norm": "rms",
        "output": "symexp_twohot", "outscale": 0.0,
        "winit": "trunc_normal_in", "bins": 255
    },
    "conhead": {
        "layers": 1, "units": 256, "act": "silu", "norm": "rms",
        "output": "binary", "outscale": 1.0, "winit": "trunc_normal_in"
    },
    "policy": {
        "layers": 3, "units": 256, "act": "silu", "norm": "rms",
        "minstd": 0.1, "maxstd": 1.0, "outscale": 0.01,
        "unimix": 0.01, "winit": "trunc_normal_in"
    },
    "value": {
        "layers": 3, "units": 256, "act": "silu", "norm": "rms",
        "output": "symexp_twohot", "outscale": 0.0,
        "winit": "trunc_normal_in", "bins": 255
    },
    "policy_dist_disc": "categorical",
    "policy_dist_cont": "bounded_normal",
    "imag_last": 0,
    "imag_length": 15,
    "horizon": 333,
    "contdisc": True,
    "imag_loss": {"slowtar": False, "lam": 0.95, "actent": 0.0003, "slowreg": 1.0},
    "repl_loss": {"slowtar": False, "lam": 0.95, "slowreg": 1.0},
    "slowvalue": {"rate": 0.02, "every": 1},
    "retnorm": {"impl": "perc", "rate": 0.01, "limit": 1.0, "perclo": 5.0, "perchi": 95.0, "debias": False},
    "valnorm": {"impl": "none", "rate": 0.01, "limit": 1e-08},
    "advnorm": {"impl": "none", "rate": 0.01, "limit": 1e-08},
    "reward_grad": True,
    "repval_loss": True,
    "repval_grad": True,
    "report": True,
    "report_gradnorms": False,
    "seed": 0,
    "logdir": "",
    "jax": {
        "platform": "cuda", "compute_dtype": "bfloat16",
        "policy_devices": [0], "train_devices": [0],
        "mock_devices": 0, "prealloc": True, "jit": True,
        "debug": False, "expect_devices": 0, "enable_policy": True,
        "coordinator_address": ""
    },
    "batch_size": 16,
    "batch_length": 64,
    "replay_context": 1,
    "report_length": 32,
    "replica": 0,
    "replicas": 1
}

tasks = [
    {
        "env_id": "highway-v0", 
        "ckpt_path": "checkpoints/highway_highway", 
        "duration": 100,
        "gif_name": "highway.gif"
    },
    {
        "env_id": "highway-fast-v0", 
        "ckpt_path": "checkpoints/highway_train_v1", 
        "duration": 100,
        "gif_name": "highway_fast.gif"
    },
    {
        "env_id": "roundabout-v0", 
        "ckpt_path": "checkpoints/highway_roundabout", 
        "duration": 100,
        "gif_name": "roundabout.gif"
    },
    {
        "env_id": "intersection-v0", 
        "ckpt_path": "checkpoints/highway_intersection", 
        "duration": 100,
        "gif_name": "intersection.gif"
    },
    {
        "env_id": "merge-v0", 
        "ckpt_path": "checkpoints/highway_merge", 
        "duration": 100,
        "gif_name": "merge.gif"
    },
]

In [ ]:
import os
import sys
import pathlib
import imageio
import numpy as np
import gymnasium as gym
import highway_env
import elements
from huggingface_hub import snapshot_download
from dreamerv3.agent import Agent
from IPython.display import Image, display

sys.path.insert(0, '/mnt/hdd/hyeonseo/workspace/dreamerv3')

os.environ['CUDA_VISIBLE_DEVICES'] = '0'
os.environ['XLA_PYTHON_CLIENT_PREALLOCATE'] = 'false'

os.environ['JAX_PLATFORMS'] = 'cuda'

import jax
print('JAX devices:', jax.devices())

REPO_ID = "HyunseoYun/dreamerv3-custom-envs"
DOWNLOAD_DIR = "./dreamerv3_checkpoints"

print(f"Hugging Face에서 체크포인트 다운로드 중... (로그인 필요 없음)")
# snapshot_download(repo_id=REPO_ID, local_dir=DOWNLOAD_DIR)
print("이전 완료!")

for task in tasks:
    env_id = task["env_id"]

    env = gym.make(env_id, render_mode='rgb_array', config={"duration": task["duration"]})

    obs_space = {
        "obs": elements.Space(np.float32, env.observation_space.shape),
        "reward": elements.Space(np.float32),
        "is_first": elements.Space(bool),
        "is_last": elements.Space(bool),
        "is_terminal": elements.Space(bool),
    }
    act_space = {"action": elements.Space(np.int32, (), 0, env.action_space.n)}

    current_config = elements.Config(config_dict).update(logdir=f"logdir/{env_id}")

    agent = Agent(obs_space, act_space, current_config)

    target_ckpt_dir = pathlib.Path(DOWNLOAD_DIR) / task["ckpt_path"]
    cp = elements.Checkpoint(target_ckpt_dir)
    cp.agent = agent

    try:
        cp.load()
        print(f"{env_id} 체크포인트 로드 완료: {target_ckpt_dir}")
    except Exception as e:
        print(f"{env_id} 로드 실패 (다음 환경으로 넘어갑니다): {e}")
        env.close()
        continue

    frames = []
    obs_raw, _ = env.reset()
    done = False
    total_reward = 0
    carry = agent.init_policy(1)
    is_first = True

    while not done:
        frame = env.render()
        frames.append(frame)

        obs = {
            "obs": np.array([obs_raw], dtype=np.float32),
            "reward": np.array([0.0], dtype=np.float32),
            "is_first": np.array([is_first]),
            "is_last": np.array([False]),
            "is_terminal": np.array([False]),
        }
        is_first = False

        carry, act, _ = agent.policy(carry, obs, mode='eval')
        action = int(act['action'][0])
        obs_raw, reward, terminated, truncated, _ = env.step(action)
        total_reward += reward
        done = terminated or truncated

    env.close()
    print(f"총 점수: {total_reward:.2f}, 프레임 수: {len(frames)}")

    imageio.mimsave(task["gif_name"], frames, fps=10)
    print("GIF 저장 완료!")

/mnt/hdd/hyeonseo/.conda/envs/dreamerv3/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


JAX devices: [cuda(id=0)]
Hugging Face에서 체크포인트 다운로드 중... (로그인 필요 없음)
이전 완료!
Observations
  obs              Space(float32, shape=(5, 5), low=-inf, high=inf)
  reward           Space(float32, shape=(), low=-inf, high=inf)
  is_first         Space(bool, shape=(), low=False, high=True)
  is_last          Space(bool, shape=(), low=False, high=True)
  is_terminal      Space(bool, shape=(), low=False, high=True)
Actions
  action           Space(int32, shape=(), low=0, high=5)
Extras
  consec           Space(int32, shape=(), low=-2147483648, high=2147483647)
  stepid           Space(uint8, shape=(20,), low=0, high=255)
  dyn/deter        Space(float32, shape=(2048,), low=-inf, high=inf)
  dyn/stoch        Space(float32, shape=(32, 16), low=-inf, high=inf)
JAX devices (1): [cuda:0]
Policy devices: cuda:0
Train devices:  cuda:0
Initializing parameters...
Optimizer opt has 9,736,477 params:
     5,782,784 dyn
       853,503 val
       794,393 dec
       789,253 pol
       721,407 rew
       656,

ALSA lib confmisc.c:855:(parse_card) cannot find card '0'
ALSA lib conf.c:5178:(_snd_config_evaluate) function snd_func_card_inum returned error: No such file or directory
ALSA lib confmisc.c:422:(snd_func_concat) error evaluating strings
ALSA lib conf.c:5178:(_snd_config_evaluate) function snd_func_concat returned error: No such file or directory
ALSA lib confmisc.c:1334:(snd_func_refer) error evaluating name
ALSA lib conf.c:5178:(_snd_config_evaluate) function snd_func_refer returned error: No such file or directory
ALSA lib conf.c:5701:(snd_config_expand) Evaluate error: No such file or directory
ALSA lib pcm.c:2664:(snd_pcm_open_noupdate) Unknown PCM default


총 점수: 85.54, 프레임 수: 100
GIF 저장 완료!
Observations
  obs              Space(float32, shape=(15, 7), low=-inf, high=inf)
  reward           Space(float32, shape=(), low=-inf, high=inf)
  is_first         Space(bool, shape=(), low=False, high=True)
  is_last          Space(bool, shape=(), low=False, high=True)
  is_terminal      Space(bool, shape=(), low=False, high=True)
Actions
  action           Space(int32, shape=(), low=0, high=3)
Extras
  consec           Space(int32, shape=(), low=-2147483648, high=2147483647)
  stepid           Space(uint8, shape=(20,), low=0, high=255)
  dyn/deter        Space(float32, shape=(2048,), low=-inf, high=inf)
  dyn/stoch        Space(float32, shape=(32, 16), low=-inf, high=inf)
JAX devices (1): [cuda:0]
Policy devices: cuda:0
Train devices:  cuda:0
Initializing parameters...


/mnt/hdd/hyeonseo/.conda/envs/dreamerv3/lib/python3.11/site-packages/gymnasium/envs/registration.py:513: DeprecationWarning: WARN: The environment intersection-v0 is out of date. You should consider upgrading to version `v1`.
  logger.deprecation(


Optimizer opt has 9,776,491 params:
     5,782,272 dyn
       853,503 val
       814,953 dec
       788,739 pol
       721,407 rew
       656,129 con
       159,488 enc
Done initializing!
Compiling 1 checkpoint groups...
Largest checkpoint group: 0 GB
Compiling train and report...
Train cost analysis:
  FLOPS:            3.1e+09
  Memory (temp):    6.3e+08
  Memory (inputs):  1.3e+08
  Memory (outputs): 1.3e+08
  Memory (code):    4.7e+06

Report cost analysis:
  FLOPS:            8.2e+06
  Memory (temp):    5.2e+06
  Memory (inputs):  2.7e+07
  Memory (outputs): 8.2e+04
  Memory (code):    1.0e+05

Done compiling!
Loading checkpoint: dreamerv3_checkpoints/checkpoints/highway_intersection/20260609T130135F708342
Loaded checkpoint.
intersection-v0 체크포인트 로드 완료: dreamerv3_checkpoints/checkpoints/highway_intersection


ALSA lib confmisc.c:855:(parse_card) cannot find card '0'
ALSA lib conf.c:5178:(_snd_config_evaluate) function snd_func_card_inum returned error: No such file or directory
ALSA lib confmisc.c:422:(snd_func_concat) error evaluating strings
ALSA lib conf.c:5178:(_snd_config_evaluate) function snd_func_concat returned error: No such file or directory
ALSA lib confmisc.c:1334:(snd_func_refer) error evaluating name
ALSA lib conf.c:5178:(_snd_config_evaluate) function snd_func_refer returned error: No such file or directory
ALSA lib conf.c:5701:(snd_config_expand) Evaluate error: No such file or directory
ALSA lib pcm.c:2664:(snd_pcm_open_noupdate) Unknown PCM default


총 점수: 9.00, 프레임 수: 9
GIF 저장 완료!
Observations
  obs              Space(float32, shape=(5, 5), low=-inf, high=inf)
  reward           Space(float32, shape=(), low=-inf, high=inf)
  is_first         Space(bool, shape=(), low=False, high=True)
  is_last          Space(bool, shape=(), low=False, high=True)
  is_terminal      Space(bool, shape=(), low=False, high=True)
Actions
  action           Space(int32, shape=(), low=0, high=5)
Extras
  consec           Space(int32, shape=(), low=-2147483648, high=2147483647)
  stepid           Space(uint8, shape=(20,), low=0, high=255)
  dyn/deter        Space(float32, shape=(2048,), low=-inf, high=inf)
  dyn/stoch        Space(float32, shape=(32, 16), low=-inf, high=inf)
JAX devices (1): [cuda:0]
Policy devices: cuda:0
Train devices:  cuda:0
Initializing parameters...
Optimizer opt has 9,736,477 params:
     5,782,784 dyn
       853,503 val
       794,393 dec
       789,253 pol
       721,407 rew
       656,129 con
       139,008 enc
Done initializing

ALSA lib confmisc.c:855:(parse_card) cannot find card '0'
ALSA lib conf.c:5178:(_snd_config_evaluate) function snd_func_card_inum returned error: No such file or directory
ALSA lib confmisc.c:422:(snd_func_concat) error evaluating strings
ALSA lib conf.c:5178:(_snd_config_evaluate) function snd_func_concat returned error: No such file or directory
ALSA lib confmisc.c:1334:(snd_func_refer) error evaluating name
ALSA lib conf.c:5178:(_snd_config_evaluate) function snd_func_refer returned error: No such file or directory
ALSA lib conf.c:5701:(snd_config_expand) Evaluate error: No such file or directory
ALSA lib pcm.c:2664:(snd_pcm_open_noupdate) Unknown PCM default


총 점수: 15.19, 프레임 수: 17
GIF 저장 완료!


시각화